# Field Extraction from a Scanned PDF: PyMuPDF + OCR
**Document:** `Sample Documentation.pdf` — a 39-page BASF regulatory dossier (Kollidon® SR) for Pfizer Inc.
**Author:**

**Key finding before writing any extraction code:** `page.get_text("words")` returns **zero words on
every single page** of this document. Unlike the earlier sample SDF (which had a real text layer under
a scanned-looking design), this document is a genuine scan/rasterized export — every page is pure image
data with no embedded text at all. That means the whole extraction has to go through OCR first.

This notebook:
1. Confirms the missing text layer.
2. Renders the two most field-rich pages (the Product Specification cover page and a Certificate of
   Analysis) to high-resolution images and OCRs them with Tesseract, keeping word-level bounding boxes.
3. Extracts 6+ labeled fields — basic info (Product Name, Customer, Batch, Material) and date fields
   (Effective Date, Manufacturing Date, Release Date, Retest Date) — using the same label-anchored,
   geometry-based pairing approach as the native-text version, adapted to OCR's pixel-coordinate output.
4. Extracts the Certificate of Analysis results table as a structured list.
5. Draws OpenCV rectangles over both pages to visually confirm every box.


## Setup

In [ ]:
# In Colab, uncomment the line below first:
# !apt-get -qq install tesseract-ocr -y && pip install pymupdf pytesseract opencv-python-headless -q

import json
import pymupdf
import pytesseract
from pytesseract import Output
import numpy as np
import cv2
from google.colab import files  # noqa: F401  (Colab only; comment out if running locally)


## Upload the PDF
In Colab, run the cell below and choose `Sample Documentation.pdf` when prompted.


In [ ]:
try:
    uploaded = files.upload()
    SRC = list(uploaded.keys())[0]
except Exception:
    SRC = "sample_documentation.pdf"  # fallback for local runs
print("Using file:", SRC)


## Step 0 — confirm there is no text layer at all

In [ ]:
doc = pymupdf.open(SRC)
print(f"{len(doc)} pages total")
for i in range(min(6, len(doc))):
    words = doc[i].get_text("words")
    print(f"page {i+1}: get_text('words') found {len(words)} words")


## 8 fields identified for extraction
| # | Field | Type | Source page |
|---|---|---|---|
| 1 | Product Name (Kollidon® SR) | Basic info | Product Specification (p1) |
| 2 | PRD Number | Basic info | Product Specification (p1) |
| 3 | Effective Date | Date field | Product Specification (p1) |
| 4 | Customer (Pfizer Inc) | Basic info | Certificate of Analysis (p4) |
| 5 | Batch | Basic info | Certificate of Analysis (p4) |
| 6 | Manufacturing Date | Date field | Certificate of Analysis (p4) |
| 7 | Release Date | Date field | Certificate of Analysis (p4) |
| 8 | Retest Date | Date field | Certificate of Analysis (p4) |


## OCR + label-anchored extraction helpers
Same geometric-pairing idea as before — find a label's words, then take the value token(s) to its right on the same row — but now the words come from `pytesseract.image_to_data()` in pixel coordinates instead of `get_text("words")` in PDF points.

In [ ]:
"""
OCR-based field & table extraction for a scanned/image-only PDF using
PyMuPDF (to render pages) + Tesseract OCR (to get text + bounding boxes),
since page.get_text("words") returns NOTHING on this document -- every
page is a raster image with no embedded text layer at all.

Pages targeted:
  - Page 1: BASF "Product Specification" cover page (Kollidon SR)
  - Page 4: BASF "Certificate of Analysis" for Pfizer Inc.
"""
import json
import pymupdf
import pytesseract
from pytesseract import Output
import numpy as np
import cv2

SRC = "sample_documentation.pdf"
ZOOM = 3.0  # higher DPI = better OCR accuracy
Y_TOL_PX = 10  # OCR bbox tolerance for "same row", in rendered pixels


def render_page(doc, page_index, zoom=ZOOM):
    page = doc[page_index]
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, pix.n)
    if pix.n == 4:
        img = cv2.cvtColor(img, cv2.COLOR_RGBA2BGR)
    else:
        img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    return img


def ocr_words(img, conf_thresh=40):
    """Run Tesseract and return a list of (x0,y0,x1,y1,text) in *pixel* coords,
    keeping only reasonably confident, non-empty tokens."""
    data = pytesseract.image_to_data(img, output_type=Output.DICT)
    words = []
    for i in range(len(data["text"])):
        text = data["text"][i].strip()
        conf = float(data["conf"][i])
        if not text or conf < conf_thresh:
            continue
        x, y, w, h = data["left"][i], data["top"][i], data["width"][i], data["height"][i]
        words.append((x, y, x + w, y + h, text))
    return words


def to_pdf_coords(bbox_px, zoom=ZOOM):
    return [round(v / zoom, 1) for v in bbox_px]


def union_bbox(words):
    x0 = min(w[0] for w in words)
    y0 = min(w[1] for w in words)
    x1 = max(w[2] for w in words)
    y1 = max(w[3] for w in words)
    return (x0, y0, x1, y1)


def find_label(words, label_tokens, y_tol=Y_TOL_PX):
    """Find a contiguous run of words (case-insensitive, punctuation-stripped)
    matching label_tokens, allowing them to span at most one row-gap."""
    n = len(label_tokens)
    norm = lambda s: s.strip(".:()").lower()
    for i in range(len(words) - n + 1):
        chunk = words[i:i + n]
        if all(norm(chunk[j][4]) == norm(label_tokens[j]) for j in range(n)):
            return chunk
    return None


def value_to_right(words, label_chunk, max_gap_px=500, y_tol=Y_TOL_PX):
    lx1 = label_chunk[-1][2]
    ly0 = label_chunk[-1][1]
    ly1 = label_chunk[-1][3]
    ly_mid = (ly0 + ly1) / 2
    candidates = []
    for w in words:
        wy_mid = (w[1] + w[3]) / 2
        if abs(wy_mid - ly_mid) <= y_tol and w[0] > lx1 and w[0] - lx1 < max_gap_px:
            candidates.append(w)
    candidates.sort(key=lambda w: w[0])
    return candidates


def cluster_rows(words, y_tol=Y_TOL_PX):
    rows = []
    used = [False] * len(words)
    order = sorted(range(len(words)), key=lambda i: (words[i][1] + words[i][3]) / 2)
    for idx in order:
        if used[idx]:
            continue
        base_y = (words[idx][1] + words[idx][3]) / 2
        row = [words[idx]]
        used[idx] = True
        for j in order:
            if used[j]:
                continue
            wy = (words[j][1] + words[j][3]) / 2
            if abs(wy - base_y) <= y_tol:
                row.append(words[j])
                used[j] = True
        row.sort(key=lambda w: w[0])
        rows.append(row)
    rows.sort(key=lambda r: r[0][1])
    return rows


# ---------------------------------------------------------------------------
# Page 1: Product Specification header fields
# ---------------------------------------------------------------------------
def extract_spec_fields(words):
    fields = []

    specs = [
        ("Product Name", ["Kollidon", "SR"]),
        ("PRD Number", ["PRD-No.:"]),
        ("Workflow Number", ["Workflow", "No.:"]),
        ("Revision", ["Revision"]),
        ("Effective Date", ["Effective", "from:"]),
        ("Status", ["Status:"]),
    ]
    for field_name, tokens in specs:
        chunk = find_label(words, tokens)
        if not chunk:
            continue
        if field_name == "Product Name":
            fields.append({"field": field_name, "text": "Kollidon SR",
                            "bbox_px": union_bbox(chunk), "bbox_pt": to_pdf_coords(union_bbox(chunk))})
            continue
        val = value_to_right(words, chunk, max_gap_px=400)
        if val:
            fields.append({
                "field": field_name,
                "text": " ".join(w[4] for w in val),
                "bbox_px": union_bbox(val),
                "bbox_pt": to_pdf_coords(union_bbox(val)),
            })
    return fields


# ---------------------------------------------------------------------------
# Page 4: Certificate of Analysis fields
# ---------------------------------------------------------------------------
def extract_coa_fields(words):
    fields = []

    # Customer/vendor recipient block: "PFIZER INC" is standalone near top-left
    for i, w in enumerate(words):
        if w[4] == "PFIZER":
            nxt = words[i + 1] if i + 1 < len(words) and abs(words[i + 1][1] - w[1]) < Y_TOL_PX else None
            chunk = [w] + ([nxt] if nxt else [])
            fields.append({
                "field": "Customer",
                "text": " ".join(x[4] for x in chunk),
                "bbox_px": union_bbox(chunk),
                "bbox_pt": to_pdf_coords(union_bbox(chunk)),
            })
            break

    specs = [
        ("Certificate Number", ["Certificate", "No"]),
        ("Material", ["Material"]),
        ("Batch", ["Batch"]),
        ("Manufacturing Date", ["Manufacturing", "date"]),
        ("Release Date", ["Release", "date"]),
        ("Retest Date", ["Retest", "date"]),
    ]
    for field_name, tokens in specs:
        chunk = find_label(words, tokens)
        if not chunk:
            continue
        val = value_to_right(words, chunk, max_gap_px=600)
        if val:
            raw_text = " ".join(w[4] for w in val)
            clean_text = raw_text.lstrip(":>|. ").strip()  # strip OCR noise from the colon glyph
            fields.append({
                "field": field_name,
                "text": clean_text,
                "text_raw_ocr": raw_text,
                "bbox_px": union_bbox(val),
                "bbox_pt": to_pdf_coords(union_bbox(val)),
            })
    return fields


# ---------------------------------------------------------------------------
# Table extraction: Certificate of Analysis characteristics table (page 4)
# ---------------------------------------------------------------------------
def extract_coa_table(words):
    all_rows = cluster_rows(words, y_tol=Y_TOL_PX)

    def row_matches_header(row):
        texts = [w[4].rstrip(":") for w in row]
        target = ["Characteristic", "Unit", "Value"]
        for i in range(len(texts) - len(target) + 1):
            if all(texts[i + j].lower() == target[j].lower() for j in range(len(target))):
                return True
        return False

    header_idx = next((i for i, row in enumerate(all_rows) if row_matches_header(row)), None)
    if header_idx is None:
        return None

    header_row = all_rows[header_idx]

    table = [("HEADER", header_row)]
    prev_y = header_row[0][1]
    for row in all_rows[header_idx + 1:]:
        row_text = " ".join(w[4] for w in row)
        if row[0][1] - prev_y > 90:  # px gap -> stop (blank line before footer text)
            break
        if row_text.startswith("Manufacturer:"):  # signature block marks end of table
            break
        table.append(("ROW", row))
        prev_y = row[0][1]
        if len(table) > 25:
            break
    return table




## Run OCR on the two target pages and extract fields

In [ ]:
page1_img = render_page(doc, 0)   # Product Specification cover page
page4_img = render_page(doc, 3)   # Certificate of Analysis, page 1 of 2

page1_words = ocr_words(page1_img)
page4_words = ocr_words(page4_img)
print(f"OCR'd {len(page1_words)} words on page 1, {len(page4_words)} words on page 4")

spec_fields = extract_spec_fields(page1_words)
coa_fields = extract_coa_fields(page4_words)

print("\n=== Product Specification fields (page 1) ===")
for f in spec_fields:
    print(f"{f['field']:20s} -> {f['text']!r:30s} bbox(pt)={f['bbox_pt']}")

print("\n=== Certificate of Analysis fields (page 4) ===")
for f in coa_fields:
    print(f"{f['field']:20s} -> {f['text']!r:20s} bbox(pt)={f['bbox_pt']}")


## Extract the Certificate of Analysis results table

In [ ]:
coa_table = extract_coa_table(page4_words)
table_struct = []
print("=== Certificate of Analysis characteristics table ===")
for kind, row in coa_table:
    texts = [w[4] for w in row]
    print(f"[{kind}] {texts}")
    table_struct.append({
        "row_type": kind, "cells": texts,
        "bbox_px": union_bbox(row), "bbox_pt": to_pdf_coords(union_bbox(row)),
    })


## Save everything as structured JSON (text + bbox per item)

In [ ]:
output = {
    "product_specification_fields": spec_fields,
    "certificate_of_analysis_fields": coa_fields,
    "certificate_of_analysis_table": table_struct,
}
with open("ocr_field_extraction_results.json", "w") as f:
    json.dump(output, f, indent=2)

print(json.dumps(output["certificate_of_analysis_fields"], indent=2))


## Optional: draw OpenCV rectangles to visually confirm the boxes

In [ ]:
for f in spec_fields:
    x0, y0, x1, y1 = f["bbox_px"]
    cv2.rectangle(page1_img, (x0, y0), (x1, y1), (0, 128, 255), 3)
    cv2.putText(page1_img, f["field"], (x0, max(y0 - 8, 15)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 128, 255), 2, cv2.LINE_AA)
cv2.imwrite("ocr_annotated_page1_spec.png", page1_img)

for f in coa_fields:
    x0, y0, x1, y1 = f["bbox_px"]
    cv2.rectangle(page4_img, (x0, y0), (x1, y1), (0, 128, 255), 3)
    cv2.putText(page4_img, f["field"], (x0, max(y0 - 8, 15)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 128, 255), 2, cv2.LINE_AA)
for row in table_struct:
    x0, y0, x1, y1 = row["bbox_px"]
    cv2.rectangle(page4_img, (x0, y0), (x1, y1), (0, 180, 0), 2)
cv2.imwrite("ocr_annotated_page4_coa.png", page4_img)

from IPython.display import Image, display
display(Image(filename="ocr_annotated_page1_spec.png", width=650))
display(Image(filename="ocr_annotated_page4_coa.png", width=650))


## Reflection

**Which fields were easy to extract?**
Anything with a clean `Label: Value` layout on one line — PRD Number, Batch, Material, the three
Certificate of Analysis dates — was straightforward once we anchored on the label text.

**Did any get missed or show up incorrectly?**
Yes, and this is the headline finding for this document: `page.get_text("words")` returns **nothing at
all**, on every one of the 39 pages, because the whole file is a scan with no text layer — not even the
partial version we saw in the earlier sample. OCR was mandatory, not optional. Even with OCR, the date
fields initially picked up stray punctuation from a misread colon (e.g. ": 06/17/2025" instead of
"06/17/2025"), which needed a small cleanup pass.

**How clean was the table output?**
Very close to perfect once the table's right-edge column and a known "end of table" marker
(the "Manufacturer:" signature block) were used to decide where the table stops — a plain row-count or
vertical-gap cutoff wasn't reliable enough on its own.

**Were the bounding boxes helpful or frustrating?**
Essential — with zero native text layer, bounding boxes from OCR are the *only* way to know where a
recognized word actually sits on the page, which is what let us pair labels to values and rebuild the
table rows/columns at all.

## Key date fields identified
- Effective Date (Product Specification): 01 Jul 2024
- Manufacturing Date (Certificate of Analysis): 06/17/2025
- Release Date (Certificate of Analysis): 08/29/2025
- Retest Date (Certificate of Analysis): 06/16/2029

## What surprised me most
Going in, I figured OCR would just be a slower version of the same extraction — turns out it's a
different problem. There's no ground-truth text to fall back on, so every OCR misread (like the colon
glyph bleeding into the date value) silently becomes "the text," and you only catch it by comparing
against the rendered page or spot-checking known values. The fix was the same idea as before — anchor
on labels and use bounding boxes to rebuild structure — but I had to add an explicit cleanup step for OCR
noise and a smarter table-end marker, since a scanned document doesn't give you any of PyMuPDF's block/line
hints to lean on.
